# Employee Shift Schedule Pipeline

Loads the live Google Sheets schedule, flattens it to one row per employee-shift,
and writes an idempotent snapshot to .

**Run on the 13th** for the first-half snapshot (days 1–15).  
**Run on the 27th** for the second-half snapshot (days 16–end).

Half is auto-detected from today's date — no manual config needed.

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
# Update the GIDs if each half uses a different Google Sheet tab

SHEET_BASE = (
    "https://docs.google.com/spreadsheets/d/e/"
    "2PACX-1vQEUW3vd8VtYtI7vy_wpMeATDMZDuW5-y4u7jmyw0qlEaBSZ8fBdNnFKMl1yTwJmQ8mRVC2jvE812b9"
    "/pub"
)

SHEET_GID = {
    1: "1943990106",   # first-half tab  (snapshot taken ~13th)
    2: "1943990106",   # second-half tab (snapshot taken ~27th) — update gid if different tab
}

TARGET_TABLE = "workspace.default.employee_schedule_snapshot"

In [ ]:
# ── Detect snapshot half from today's date ───────────────────────────────
from datetime import date

today          = date.today()
snapshot_year  = today.year
snapshot_month = today.month
snapshot_half  = 1 if today.day <= 15 else 2

gid = SHEET_GID[snapshot_half]
url = f"{SHEET_BASE}?gid={gid}&single=true&output=csv"

print(f"Today          : {today}")
print(f"Snapshot       : {snapshot_year}-{snapshot_month:02d}  half {snapshot_half}")
print(f"Sheet URL      : {url}")

In [ ]:
# ── Load & transform ─────────────────────────────────────────────────────
import pandas as pd

raw = pd.read_csv(url)
print(f"Raw sheet shape: {raw.shape}  columns: {list(raw.columns)}")

melt_cols = [col for col in raw.columns if col != 'Дата']
flat_df = raw.melt(
    id_vars='Дата',
    value_vars=melt_cols,
    var_name='store_shift',
    value_name='employee'
)

# Drop empty / whitespace-only employee cells
flat_df = flat_df[
    flat_df['employee'].notna() &
    (flat_df['employee'].astype(str).str.strip() != '')
]

# Split on the LAST " - " so store names containing " - " are handled correctly
split = flat_df['store_shift'].str.rsplit(' - ', n=1, expand=True)
flat_df['store'] = split[0].str.strip()
flat_df['shift'] = split[1].str.strip()

# Parse date (dayfirst=True for Russian DD.MM.YYYY format)
flat_df['date'] = pd.to_datetime(
    flat_df['Дата'], dayfirst=True, errors='coerce'
).dt.date

# Attach snapshot metadata
flat_df['snapshot_year']  = snapshot_year
flat_df['snapshot_month'] = snapshot_month
flat_df['snapshot_half']  = snapshot_half

flat_df = (
    flat_df[['date', 'store', 'shift', 'employee',
              'snapshot_year', 'snapshot_month', 'snapshot_half']]
    .reset_index(drop=True)
)

print(f"Rows to write  : {len(flat_df):,}")
print(f"Stores found   : {sorted(flat_df['store'].dropna().unique())}")
display(flat_df.head(20))

In [ ]:
# ── Idempotent save (delete this snapshot, then append) ─────────────────
# Running the notebook twice on the same day will NOT create duplicates.

try:
    spark.sql(f"""
        DELETE FROM {TARGET_TABLE}
        WHERE snapshot_year  = {snapshot_year}
          AND snapshot_month = {snapshot_month}
          AND snapshot_half  = {snapshot_half}
    """)
    print(f"Deleted existing rows for {snapshot_year}-{snapshot_month:02d} half {snapshot_half}")
except Exception as e:
    print(f"Delete skipped (table may not exist yet): {e}")

spark.createDataFrame(flat_df).write.mode("append").saveAsTable(TARGET_TABLE)
print(f"Written {len(flat_df):,} rows → {TARGET_TABLE}")